## 1. Install libraries

In [1]:
!pip -q install beautifulsoup4 feedparser google-genai spacy pandas requests lxml
!python -m spacy download en_core_web_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 2. Connect Google Drive

In [30]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
output_dir = Path('/content/drive/MyDrive/Assignment2')
output_dir.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Set up the API and SEC contact

In [31]:
import os
from getpass import getpass
from google import genai

SEC_EMAIL = input('Email for SEC requests: ').strip()
api_key = getpass('Gemini API key: ')
client = genai.Client(api_key=api_key)



Email for SEC requests: darinka.townsend@gmail.com
Gemini API key: ··········


In [32]:
MODEL_NAME = 'gemini-3.8-flash'

## 4. Choose companies and filing count

In [34]:
TICKERS = [
    'AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN',
    'META', 'TSLA', 'AMD', 'INTC', 'QCOM',
    'CSCO', 'ADBE', 'CRM', 'ORCL', 'IBM',
    'DELL', 'HPQ', 'AVGO', 'TXN', 'AMAT'
]

FILINGS_PER_COMPANY = 10
MAX_DOCUMENTS = 100


import requests
import time
import re
import json
import pandas as pd
import feedparser
import spacy
from bs4 import BeautifulSoup
from urllib.parse import urljoin

session = requests.Session()
session.headers.update({
    'User-Agent': f'Student Research Project {SEC_EMAIL}',
    'Accept-Encoding': 'gzip, deflate',
    'Host': 'www.sec.gov'
})
nlp = spacy.load('en_core_web_sm')

def sec_get(url):
    time.sleep(0.2)
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return response


## 5. Find CIK numbers

In [35]:
ticker_data = sec_get('https://www.sec.gov/files/company_tickers.json').json()
companies = {
    item['ticker']: {'cik': str(item['cik_str']).zfill(10), 'name': item['title']}
    for item in ticker_data.values()
}

for ticker in TICKERS:
    print(ticker, companies.get(ticker, 'Not found'))

AAPL {'cik': '0000320193', 'name': 'Apple Inc.'}
MSFT {'cik': '0000789019', 'name': 'MICROSOFT CORP'}
NVDA {'cik': '0001045810', 'name': 'NVIDIA CORP'}
GOOGL {'cik': '0001652044', 'name': 'Alphabet Inc.'}
AMZN {'cik': '0001018724', 'name': 'AMAZON COM INC'}
META {'cik': '0001326801', 'name': 'Meta Platforms, Inc.'}
TSLA {'cik': '0001318605', 'name': 'Tesla, Inc.'}
AMD {'cik': '0000002488', 'name': 'ADVANCED MICRO DEVICES INC'}
INTC {'cik': '0000050863', 'name': 'INTEL CORP'}
QCOM {'cik': '0000804328', 'name': 'QUALCOMM INC/DE'}
CSCO {'cik': '0000858877', 'name': 'CISCO SYSTEMS, INC.'}
ADBE {'cik': '0000796343', 'name': 'ADOBE INC.'}
CRM {'cik': '0001108524', 'name': 'Salesforce, Inc.'}
ORCL {'cik': '0001341439', 'name': 'ORACLE CORP'}
IBM {'cik': '0000051143', 'name': 'INTERNATIONAL BUSINESS MACHINES CORP'}
DELL {'cik': '0001571996', 'name': 'Dell Technologies Inc.'}
HPQ {'cik': '0000047217', 'name': 'HP INC'}
AVGO {'cik': '0001730168', 'name': 'Broadcom Inc.'}
TXN {'cik': '0000097476'

## 6. Get the 8-K filing links

In [36]:
def get_filings(ticker, count=20):
    cik = companies[ticker]['cik']
    url = (f'https://www.sec.gov/cgi-bin/browse-edgar?'
           f'action=getcompany&CIK={cik}&type=8-K&count={count}&output=atom')
    feed = feedparser.parse(sec_get(url).content)
    filings = []
    for entry in feed.entries:
        # The Atom feed may also include 8-K/A amendments.
        form = entry.get('category', '')
        if form and form != '8-K':
            continue
        filings.append({
            'ticker': ticker,
            'company_name': companies[ticker]['name'],
            'filing_time': entry.get('updated', entry.get('published', '')),
            'index_url': entry.get('link', '')
        })
    return filings

filings = []
for ticker in TICKERS:
    try:
        filings.extend(get_filings(ticker, FILINGS_PER_COMPANY))
        print(ticker, 'completed')
    except Exception as e:
        print(ticker, 'failed:', e)

filings = filings[:MAX_DOCUMENTS]
print('Filings found:', len(filings))

AAPL completed
MSFT completed
NVDA completed
GOOGL completed
AMZN completed
META completed
TSLA completed
AMD completed
INTC completed
QCOM completed
CSCO completed
ADBE completed
CRM completed
ORCL completed
IBM completed
DELL completed
HPQ completed
AVGO completed
TXN completed
AMAT completed
Filings found: 100


## 7. Download and clean filing text

In [37]:
from urllib.parse import urljoin, urlparse, parse_qs
from bs4 import BeautifulSoup

def get_filing_text(index_url):
    response = sec_get(index_url)
    soup = BeautifulSoup(response.text, 'html.parser')

    table = soup.find('table', class_='tableFile')
    if table is None:
        raise ValueError('Filing table not found')

    urls = []

    for row in table.find_all('tr'):
        cells = row.find_all('td')

        if len(cells) < 4:
            continue

        doc_type = cells[3].get_text(strip=True)
        link = cells[2].find('a')

        if not link:
            continue

        if doc_type == '8-K' or doc_type.startswith('EX-99'):
            url = urljoin(index_url, link['href'])

            # Remove the XBRL viewer
            if '/ix?doc=' in url:
                path = parse_qs(urlparse(url).query)['doc'][0]
                url = urljoin('https://www.sec.gov', path)

            urls.append(url)

    texts = []

    for url in urls:
        response = sec_get(url)
        page = BeautifulSoup(response.text, 'html.parser')

        for tag in page(['script', 'style']):
            tag.decompose()

        text = page.get_text(' ', strip=True)

        if len(text) > 500:
            texts.append(text)

    if not texts:
        raise ValueError('No valid filing text found')

    return '\n\n'.join(texts)

KEYWORDS = ['new product', 'product launch', 'introduces', 'unveils',
            'announced the launch', 'new platform', 'new service', 'debut']

def find_product_sections(text, window=700):
    snippets = []
    lower = text.lower()
    for keyword in KEYWORDS:
        for match in list(re.finditer(re.escape(keyword), lower))[:5]:
            start = max(0, match.start() - window)
            end = min(len(text), match.end() + window)
            snippets.append(text[start:end])
    return '\n---\n'.join(snippets[:12])


## 8. Extract product announcements with NER and Gemini

In [28]:
def extract_products(filing, text):
    sections = find_product_sections(text)
    if not sections:
        return []

    doc = nlp(sections[:80000])
    organizations = list(dict.fromkeys(ent.text for ent in doc.ents if ent.label_ == 'ORG'))[:15]

    prompt = f"""Read the SEC 8-K excerpts and identify genuinely NEW product or service
announcements. Do not include routine financial results, old products, or speculation.
Return only a JSON array. Each element must have keys:
new_product, product_description.
Use one element per new product, or [] if none is clearly announced.
Each description must be fewer than 180 characters.
Do not invent product names or details.

Company: {filing['company_name']}
NER organization candidates: {organizations}
Excerpts:
{sections[:25000]}"""

    response = client.models.generate_content(model=MODEL, contents=prompt,
        config={'response_mime_type': 'application/json'})
    items = json.loads(response.text)
    if not isinstance(items, list):
        return []

    results = []
    for item in items:
        name = str(item.get('new_product', '')).strip()
        description = str(item.get('product_description', '')).strip()
        if name and description and len(description) < 180:
            results.append({
                'company_name': filing['company_name'],
                'stock_name': filing['ticker'],
                'filing_time': filing['filing_time'],
                'new_product': name,
                'product_description': description
            })
    return results


## 9. Process filings and save progress

In [12]:
csv_path = output_dir / 'sec_8k_products.csv'
log_path = output_dir / 'processed_filings.csv'

columns = [
    'company_name',
    'stock_name',
    'filing_time',
    'new_product',
    'product_description'
]

if csv_path.exists():
    results = pd.read_csv(csv_path).to_dict('records')
else:
    results = []

if log_path.exists():
    processed = set(pd.read_csv(log_path)['index_url'].dropna())
else:
    processed = set()

for i, filing in enumerate(filings, 1):

    if filing['index_url'] in processed:
        continue

    try:
        text = get_filing_text(filing['index_url'])

        if not text:
            print(f'{i}: No text found')
            continue

        products = extract_products(filing, text)

        results.extend(products)
        processed.add(filing['index_url'])

        pd.DataFrame(results, columns=columns).to_csv(
            csv_path, index=False
        )

        pd.DataFrame({
            'index_url': sorted(processed)
        }).to_csv(log_path, index=False)

        print(f'{i}/{len(filings)}: {len(products)} products')

    except Exception as e:
        print(f'{i}: Error - {e}')

print('Documents processed:', len(processed))
print('Products extracted:', len(results))
print('Saved to:', csv_path)

/tmp/ipykernel_25834/1246032721.py:15: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  page = BeautifulSoup(sec_get(document_url).text, 'lxml')


1/100: 0 products


/tmp/ipykernel_25834/1246032721.py:15: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  page = BeautifulSoup(sec_get(document_url).text, 'lxml')


2/100: 0 products


/tmp/ipykernel_25834/1246032721.py:15: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  page = BeautifulSoup(sec_get(document_url).text, 'lxml')


3/100: 0 products


... (189 outputs left out)

98/100: 0 products


/tmp/ipykernel_25834/1246032721.py:15: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  page = BeautifulSoup(sec_get(document_url).text, 'lxml')


99/100: 0 products
100/100: 0 products
Documents processed: 100
Products extracted: 0
Saved to: /content/drive/MyDrive/Assignment2/sec_8k_products.csv


/tmp/ipykernel_25834/1246032721.py:15: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  page = BeautifulSoup(sec_get(document_url).text, 'lxml')


## 10. Review the results

In [13]:
df = pd.read_csv(csv_path)
display(df.head(20))
print('Products found:', len(df))
print('Companies:', df['stock_name'].nunique() if len(df) else 0)
print('Descriptions under 180 characters:', df['product_description'].fillna('').str.len().lt(180).all())

,company_name,stock_name,filing_time,new_product,product_description


Products found: 0
Companies: 0
Descriptions under 180 characters: True


In [14]:
import pandas as pd

log_path = output_dir / 'processed_filings.csv'

if log_path.exists():
    log = pd.read_csv(log_path)
    print('Documents processed:', len(log))
else:
    print('No processing log found')

print('Filings collected:', len(filings))

Documents processed: 100
Filings collected: 100


In [15]:
filing = filings[0]

print('Company:', filing['company_name'])
print('URL:', filing['index_url'])

text = get_filing_text(filing['index_url'])

print('Text length:', len(text))
print(text[:2000])

Company: Apple Inc.
URL: https://www.sec.gov/Archives/edgar/data/320193/000032019326000024/0000320193-26-000024-index.htm
Text length: 73
XBRL Viewer Please enable JavaScript to use the EDGAR Inline XBRL Viewer.


/tmp/ipykernel_25834/1246032721.py:15: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  page = BeautifulSoup(sec_get(document_url).text, 'lxml')


In [38]:
products = extract_products(filing, text)

print(products)

[]


In [39]:
filing = filings[0]

text = get_filing_text(filing['index_url'])

print('Text length:', len(text))
print(text[:2000])

Text length: 15056
aapl-20261102 false 0000320193 0000320193 2026-11-02 2026-11-02 0000320193 us-gaap:CommonStockMember 2026-11-02 2026-11-02 0000320193 aapl:A1.625NotesDue2026Member 2026-11-02 2026-11-02 0000320193 aapl:A2.000NotesDue2027Member 2026-11-02 2026-11-02 0000320193 aapl:A1.375NotesDue2029Member 2026-11-02 2026-11-02 0000320193 aapl:A3.050NotesDue2029Member 2026-11-02 2026-11-02 0000320193 aapl:A0.500Notesdue2031Member 2026-11-02 2026-11-02 0000320193 aapl:A3.600NotesDue2042Member 2026-11-02 2026-11-02 UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 8-K CURRENT REPORT Pursuant to Section 13 OR 15(d) of The Securities Exchange Act of 1934 November 2, 2026 Date of Report (Date of earliest event reported) Apple Inc. (Exact name of Registrant as specified in its charter) California 001-36743 94-2404110 (State or other jurisdiction of incorporation) (Commission File Number) (I.R.S. Employer Identification No.) One Apple Park Way MS 927-4GC Cupertino 

In [40]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin

filing = filings[0]

response = sec_get(filing['index_url'])
soup = BeautifulSoup(response.text, 'html.parser')

table = soup.find('table', class_='tableFile')

for row in table.find_all('tr'):
    cells = row.find_all('td')

    if len(cells) >= 4:
        link = cells[2].find('a')

        if link:
            print(
                cells[3].get_text(strip=True),
                urljoin(filing['index_url'], link['href'])
            )

8-K https://www.sec.gov/ix?doc=/Archives/edgar/data/320193/000032019326000024/aapl-20261102.htm
EX-99.1 https://www.sec.gov/Archives/edgar/data/320193/000032019326000024/a8-kex991q4202609262026.htm
GRAPHIC https://www.sec.gov/Archives/edgar/data/320193/000032019326000024/aapl-20261102_g1.jpg
 https://www.sec.gov/Archives/edgar/data/320193/000032019326000024/0000320193-26-000024.txt


In [41]:
for filing in filings[:5]:
    try:
        text = get_filing_text(filing['index_url'])
        products = extract_products(filing, text)

        print(filing['company_name'])
        print('Text length:', len(text))
        print('Products:', products)
        print('-' * 50)

    except Exception as e:
        print('Error:', e)

Apple Inc.
Text length: 15056
Products: []
--------------------------------------------------
Apple Inc.
Text length: 14422
Products: []
--------------------------------------------------
Error: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}
Apple Inc.
Text length: 4989
Products: []
--------------------------------------------------
Apple Inc.
Text length: 5601
Products: []
--------------------------------------------------


In [ ]:
response = client.models.generate_content(
    model='gemini-3.8-flash',
    contents='Say hello'
)

print(response.text)